In [ ]:
"""
Cell A — Exposure-normalised crash rate analysis (corrected)
============================================================
Produces Table 4 and Figures 4-6.

Denominator : total Census 2021 cycling commuters across EVERY LSOA in the study
              region assigned to a given IMD decile, including LSOAs in which no
              casualty was recorded.
Numerator   : casualties whose RESIDENTIAL LSOA lies inside the study region and
              is assigned to that decile, so that numerator and denominator
              describe the same population. IMD deciles are national, not
              region-internal, so without this restriction a region-only
              denominator is paired with a partly national numerator.
Region       : derived from the data — each casualty's crash-location LSOA is
              matched to its 2019 Local Authority District and the region is the
              full set of districts so identified.

Required input files (edit the paths below if yours differ):
  IMD_FILE     File 7 / File 1 of the English Indices of Deprivation 2019
  TRAVEL_FILE  ONS Census 2021 TS061, LSOA level
  LOOKUP_FILE  ONS Best Fit LSOA(2011) -> LSOA(2021)
  the three regional casualty point files
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# ---------------------------------------------------------------- configuration
# Files are located by pattern rather than by exact name. Government downloads
# often contain en-dashes, timestamps, or bracketed suffixes that change between
# machines, so hard-coded filenames break easily. DATA_DIR defaults to the
# notebook's own directory; point it elsewhere if the data lives in another folder.
import glob
import os

DATA_DIR = '.'
IMD_SHEET = 'IMD2019'
N_YEARS = 2          # 2023-2024

# Roots searched, in order, for every input file. The search is recursive, so a
# file sitting in a subfolder is found without any configuration; ~/Downloads is
# included because government downloads often never get moved.
SEARCH_ROOTS = [
    DATA_DIR,
    os.path.expanduser('~/Downloads'),
    os.path.expanduser('~/Desktop'),
]


def find_file(label, *patterns):
    """Find the first file matching any pattern, searching each root recursively."""
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for pattern in patterns:
            hits = [h for h in
                    sorted(glob.glob(os.path.join(root, '**', pattern), recursive=True))
                    if os.path.isfile(h)]
            if hits:
                if len(hits) > 1:
                    print(f'  NOTE: {len(hits)} files matched for {label}; using {hits[0]}')
                return hits[0]
    listing = []
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for h in sorted(glob.glob(os.path.join(root, '**', '*'), recursive=True))[:400]:
            if os.path.isfile(h) and h.lower().endswith(('.csv', '.xlsx', '.xls')):
                listing.append(os.path.relpath(h, root))
    raise FileNotFoundError(
        f'Could not find {label}.\n'
        f'  Patterns tried    : {list(patterns)}\n'
        f'  Roots searched    : {[os.path.abspath(r) for r in SEARCH_ROOTS]}\n'
        f'  Data files found  : {listing}\n'
        f'  If the file is somewhere else entirely, set the variable directly, e.g.\n'
        f"      IMD_FILE = '/full/path/to/your/file.xlsx'\n"
        f'  and re-run from the line below the find_file() calls.')


IMD_FILE    = find_file('IMD 2019 index file',
                        '*IMD2019*Multiple*Deprivation*.xlsx',
                        '*IMD*2019*.xlsx',
                        'File_1*.xlsx')
TRAVEL_FILE = find_file('Census 2021 TS061 travel-to-work file',
                        '*TS061*.csv',
                        '*travel*work*.csv')
LOOKUP_FILE = find_file('LSOA 2011-to-2021 Best Fit lookup',
                        '*lsoa11*lsoa21*lookup*.csv',
                        '*LSOA11*LSOA21*.csv',
                        '*best*fit*.csv')

REGION_FILES = {
    'Greater London':     find_file('London casualty points', 'london_point*.csv', '*london*point*.csv'),
    'Greater Manchester': find_file('Manchester casualty points', 'manchester_point*.csv', '*manchester*point*.csv'),
    'West Midlands':      find_file('Birmingham casualty points', 'birmingham_point*.csv', '*birmingham*point*.csv'),
}

print('Resolved input files (check these are the ones you expect):')
for label, path in [('IMD', IMD_FILE), ('TS061', TRAVEL_FILE), ('LSOA lookup', LOOKUP_FILE),
                    *REGION_FILES.items()]:
    print(f'  {label:<20} {os.path.abspath(path)}')
print()

# --------------------------------------------- 1. IMD: LSOA(2011) -> decile, LAD
imd = pd.read_excel(IMD_FILE, sheet_name=IMD_SHEET).rename(columns={
    'LSOA code (2011)': 'lsoa11',
    'Local Authority District name (2019)': 'lad',
    'Index of Multiple Deprivation (IMD) Decile': 'decile',
})[['lsoa11', 'lad', 'decile']]
print(f'IMD 2019: {len(imd):,} LSOAs, {imd["lad"].nunique()} local authorities')

# ------------------------------------ 2. Cycling commuters per 2011 LSOA
travel = pd.read_csv(TRAVEL_FILE)
GEOG, CAT, VAL = ('Lower layer Super Output Areas Code',
                  'Method used to travel to workplace (12 categories)',
                  'Observation')
cycling = (travel[travel[CAT].astype(str).str.strip().eq('Bicycle')][[GEOG, VAL]]
           .rename(columns={GEOG: 'LSOA21CD', VAL: 'commuters'}))

lookup = pd.read_csv(LOOKUP_FILE)
lookup.columns = [c.lstrip('\ufeff') for c in lookup.columns]   # strip BOM

# One row per 2011 LSOA. Where a 2011 LSOA was split into several 2021 LSOAs the
# commuter counts are summed back together, so joining this table onto casualty
# records cannot duplicate rows.
commuters = (lookup[['LSOA11CD', 'LSOA21CD']].drop_duplicates()
             .merge(cycling, how='left', on='LSOA21CD')
             .groupby('LSOA11CD', as_index=False)['commuters'].sum()
             .rename(columns={'LSOA11CD': 'lsoa11'}))
commuters['commuters'] = commuters['commuters'].fillna(0)
print(f'Cycling commuters: {len(commuters):,} LSOAs, '
      f'{commuters["commuters"].sum():,.0f} commuters nationally')

# ------------------------------------------------ 3. Per region: rate by decile
results, tables = {}, []

for region, path in REGION_FILES.items():
    crashes = pd.read_csv(path, low_memory=False)

    # -- region extent, derived from the crash-location LSOAs themselves --------
    lads = sorted(crashes.merge(imd, how='left',
                                left_on='lsoa_of_accident_location',
                                right_on='lsoa11')['lad'].dropna().unique())

    # -- denominator: every LSOA in those districts, crash or no crash ----------
    base = imd[imd['lad'].isin(lads)].merge(commuters, how='left', on='lsoa11')
    base['commuters'] = base['commuters'].fillna(0)
    denom = (base.groupby('decile')
             .agg(n_lsoa=('lsoa11', 'size'), commuters=('commuters', 'sum'))
             .reset_index())

    # -- numerator: casualties resident inside the region ----------------------
    resident = (crashes[crashes['lsoa_of_casualty'].astype(str) != '-1']
                .merge(imd, how='left', left_on='lsoa_of_casualty', right_on='lsoa11'))
    resident = resident[resident['lad'].isin(lads)]
    numer = (resident.groupby('decile', as_index=False).size()
             .rename(columns={'size': 'crashes'}))

    summary = denom.merge(numer, how='left', on='decile')
    summary['crashes'] = summary['crashes'].fillna(0).astype(int)
    summary['rate'] = summary['crashes'] / N_YEARS / summary['commuters'] * 1000
    summary = summary.sort_values('decile').reset_index(drop=True)
    results[region] = summary

    rho, pval = spearmanr(summary['decile'], summary['rate'])
    d1 = summary.loc[summary['decile'] == 1, 'rate'].iloc[0]
    d10 = summary.loc[summary['decile'] == 10, 'rate'].iloc[0]

    print(f'\n===== {region}')
    print(f'  {len(lads)} local authorities, {len(base):,} LSOAs, '
          f'{base["commuters"].sum():,.0f} cycling commuters')
    print(f'  casualties: {len(crashes):,} recorded in region, '
          f'{len(resident):,} resident in region '
          f'({1 - len(resident)/len(crashes):.1%} excluded)')
    print(summary.round(2).to_string(index=False))
    print(f'  D1 = {d1:.1f}   D10 = {d10:.1f}   D10/D1 = {d10/d1:.2f}x')
    print(f'  Spearman(decile, rate) = {rho:+.3f}  p = {pval:.3f}')

    # -- Figure ---------------------------------------------------------------
    plt.figure(figsize=(8, 5))
    sns.barplot(data=summary, x='decile', y='rate',
                hue='decile', palette='mako', legend=False)
    plt.title(f'Cycling-Exposure-Normalised Crash Rate by IMD Decile — {region}\n'
              f'(annual average, 2023–2024)')
    plt.xlabel('IMD Decile (1 = most deprived)')
    plt.ylabel('Crashes per 1,000 cycling commuters per year')
    plt.tight_layout()
    slug = region.lower().replace(' ', '_')
    plt.savefig(f'{slug}_exposure_normalised_rate.png', dpi=300)
    plt.show()

    t = summary.copy()
    t.insert(0, 'region', region)
    tables.append(t)

# ------------------------------------------------------------ 4. Table 4
table4 = pd.concat(tables, ignore_index=True)
table4.round(2).to_csv('table_4_exposure_normalised_rates.csv', index=False)

wide = table4.pivot(index='decile', columns='region', values='rate').round(1)
wide = wide[['Greater London', 'Greater Manchester', 'West Midlands']]
print('\n\n========== Table 4 ==========')


In [ ]:
"""
Cell B — Deprivation co-location of crash locations
===================================================
Produces Table 3.

Reported in Section 5.1 of the report. Each casualty is matched to the IMD 2019
decile of the LSOA in which the collision OCCURRED (lsoa_of_accident_location),
not the decile of the casualty's home LSOA, which is what Cell A uses. The two
analyses therefore answer different questions: this one asks where crashes
happen, Cell A asks how much risk each resident cyclist carries.

The share of casualties falling in each decile is compared against two reference
distributions, because neither is a clean exposure denominator:

  baseline 1  the share of the region's LSOAs that fall in that decile
              -> answers "are crashes over-represented relative to how much of
                 the region is deprived?"
  baseline 2  the share of the region's cycling commuters resident in LSOAs of
              that decile
              -> answers "are crashes over-represented relative to how much
                 cycling happens there?"

A representation ratio above 1.00 means casualties are over-represented in that
decile relative to the baseline. Spearman rank correlation between decile and
ratio summarises whether the over-representation runs monotonically with
deprivation.

Requires Cell A to have run first: it reuses `imd`, `commuters` and
`REGION_FILES` from that cell.
"""

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# Cell A must have run: these objects are reused rather than rebuilt so that the
# two analyses are guaranteed to use identical deprivation and exposure inputs.
for _name in ('imd', 'commuters', 'REGION_FILES'):
    if _name not in dir():
        raise NameError(f'{_name} not defined — run Cell A first.')

colocation, rho_rows = {}, {}

for region, path in REGION_FILES.items():
    crashes = pd.read_csv(path, low_memory=False)

    # -- region extent, derived exactly as in Cell A ---------------------------
    located = crashes.merge(imd, how='left',
                            left_on='lsoa_of_accident_location',
                            right_on='lsoa11')
    lads = sorted(located['lad'].dropna().unique())

    # -- numerator: casualties by CRASH-LOCATION decile ------------------------
    located = located[located['lad'].isin(lads)]
    cas = (located.groupby('decile', as_index=False).size()
           .rename(columns={'size': 'casualties'}))

    # -- baselines: every LSOA in those districts, crash or no crash -----------
    base = imd[imd['lad'].isin(lads)].merge(commuters, how='left', on='lsoa11')
    base['commuters'] = base['commuters'].fillna(0)
    ref = (base.groupby('decile')
           .agg(n_lsoa=('lsoa11', 'size'), commuters=('commuters', 'sum'))
           .reset_index())

    t = ref.merge(cas, how='left', on='decile')
    t['casualties'] = t['casualties'].fillna(0).astype(int)

    # -- shares, then ratio of shares -----------------------------------------
    t['share_casualties'] = t['casualties'] / t['casualties'].sum()
    t['share_lsoa'] = t['n_lsoa'] / t['n_lsoa'].sum()
    t['share_commuters'] = t['commuters'] / t['commuters'].sum()
    t['ratio_vs_lsoa'] = t['share_casualties'] / t['share_lsoa']
    t['ratio_vs_commuters'] = t['share_casualties'] / t['share_commuters']
    t = t.sort_values('decile').reset_index(drop=True)

    rho_l, p_l = spearmanr(t['decile'], t['ratio_vs_lsoa'])
    rho_c, p_c = spearmanr(t['decile'], t['ratio_vs_commuters'])
    colocation[region] = t
    rho_rows[region] = dict(rho_lsoa=rho_l, p_lsoa=p_l,
                            rho_commuters=rho_c, p_commuters=p_c)

    print(f'\n===== {region}')
    print(f'  {len(lads)} local authorities, {len(base):,} LSOAs, '
          f'{t["casualties"].sum():,} casualties with a valid crash-location LSOA')
    print(t[['decile', 'casualties', 'n_lsoa', 'commuters',
             'ratio_vs_lsoa', 'ratio_vs_commuters']].round(2).to_string(index=False))
    print(f'  vs share of LSOAs      : Spearman rho = {rho_l:+.2f}  p = {p_l:.3f}')
    print(f'  vs share of commuters  : Spearman rho = {rho_c:+.2f}  p = {p_c:.3f}')

# ------------------------------------------------------------------ Table 3
wide = pd.DataFrame({'Decile': range(1, 11)}).set_index('Decile')
short = {'Greater London': 'GL', 'Greater Manchester': 'GM', 'West Midlands': 'WM'}
for region, t in colocation.items():
    s = short[region]
    idx = t.set_index('decile')
    wide[f'{s} vs LSOAs'] = idx['ratio_vs_lsoa'].round(2)
    wide[f'{s} vs cyclists'] = idx['ratio_vs_commuters'].round(2)

rho_line = {}
for region in colocation:
    s = short[region]
    rho_line[f'{s} vs LSOAs'] = round(rho_rows[region]['rho_lsoa'], 2)
    rho_line[f'{s} vs cyclists'] = round(rho_rows[region]['rho_commuters'], 2)
table3 = pd.concat([wide, pd.DataFrame(rho_line, index=['Spearman rho'])])

table3.to_csv('table_3_colocation_representation_ratios.csv')
print('\n\n========== Table 3 ==========')
print(table3.to_string())
print('\np-values:')
for region, r in rho_rows.items():
    print(f'  {region:<20} vs LSOAs p = {r["p_lsoa"]:.3f}   '
          f'vs cyclists p = {r["p_commuters"]:.3f}')
print('\nSaved: table_3_colocation_representation_ratios.csv')


In [ ]:
"""
Cell C — Cycling crash severity classification
==============================================
Produces Tables 5-7, Figures 7-9 (ROC) and Figures 10-12 (SHAP).

Encoding note:
    All sixteen candidate features arrive from the CSV as int64, so
    get_dummies() finds no object columns and each STATS19 code enters the model
    on its numeric scale, the -1 "missing / unknown / not applicable" code
    included. Section 4.4 of the report states this explicitly and Appendix D
    reports the same models refitted with the code fields one-hot encoded; the
    categorical variant is reproduced by Cell D of this notebook.

Properties:
1. All three regions use the Combined Authority extracts, so the modelling and
   the exposure analysis in Cell A cover the same geography.
2. The Random Forest grid search wraps SMOTE and the classifier in an
   imbalanced-learn Pipeline, so oversampling happens inside each search fold.
3. sex_of_casualty == -1 is filtered out before modelling.
4. SHAP is run for Greater London only, for estimate stability rather than
   headline AUC.
"""

import copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, GridSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve, auc)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------- configuration
# Reuses the REGION_FILES resolved in Cell A if that cell has already run;
# otherwise resolves them the same way.
try:
    REGION_FILES
except NameError:
    import glob, os
    def _first(*pats):
        roots = ['.', os.path.expanduser('~/Downloads'), os.path.expanduser('~/Desktop')]
        for root in roots:
            for p in pats:
                hits = sorted(glob.glob(os.path.join(root, '**', p), recursive=True))
                if hits:
                    return hits[0]
        raise FileNotFoundError(f'none of {pats} found under {roots}')
    REGION_FILES = {
        'Greater London':     _first('london_point*.csv', '*london*point*.csv'),
        'Greater Manchester': _first('manchester_point*.csv', '*manchester*point*.csv'),
        'West Midlands':      _first('birmingham_point*.csv', '*birmingham*point*.csv'),
    }

SMALL_SAMPLE_THRESHOLD = 1000    # below this n, use 5-fold cross-validation

# Feature selection is a positive whitelist rather than an exclusion list: only
# the variables named here can enter the model, so every post-crash STATS19
# descriptor (pedestrian_movement, vehicle_leaving_carriageway,
# hit_object_in_carriageway, ...) is excluded by construction.
CANDIDATE_FEATURES = [
    'casualty_imd_decile',
    'speed_limit',
    'road_type',
    'junction_detail',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions',
    'day_of_week',
    'first_road_class',
    'age_of_casualty',
    'sex_of_casualty',
    'casualty_type',          # constant within a cyclist-only sample (zero variance)
    'pedestrian_location',    # not applicable to cyclist casualties (near-constant)
    'urban_or_rural_area',
    'number_of_vehicles',
    'number_of_casualties',
]

all_region_metrics = {}

for region, path in REGION_FILES.items():
    slug = region.lower().replace(' ', '_')
    print(f"\n{'='*60}\n{region}\n{'='*60}")

    df = pd.read_csv(path, low_memory=False)

    # ---------- target: STATS19 1 = Fatal, 2 = Serious, 3 = Slight ------------
    df['Y'] = np.where(df['collision_severity'] <= 2, 1, 0)

    # ---------- drop sex_of_casualty == -1 only -------------------------------
    # The STATS19 "unknown" sex code is removed rather than kept as a level,
    # because it produced attributions unlike either male or female. The -1 code
    # is retained as an explicit level for every other feature.
    if 'sex_of_casualty' in df.columns:
        before = len(df)
        df = df[df['sex_of_casualty'] != -1].reset_index(drop=True)
        print(f'Removed sex_of_casualty == -1: {before} -> {len(df)} '
              f'({before - len(df)} rows)')

    n_total = len(df)
    print(f'Sample size: {n_total};  serious/fatal share: {df["Y"].mean():.1%}')

    # ---------- leakage guard -------------------------------------------------
    feature_cols = [c for c in CANDIDATE_FEATURES if c in df.columns]
    leak = [c for c in feature_cols if 'severity' in c.lower()]
    if leak:
        raise ValueError(f'Leakage guard tripped: {leak}')

    # ---------- encoding ------------------------------------------------------
    X = df[feature_cols].copy()
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    X[cat_cols] = X[cat_cols].astype(str)
    X_encoded = pd.get_dummies(X, drop_first=True)
    y = df['Y']

    use_cv = n_total < SMALL_SAMPLE_THRESHOLD

    # ======================================================================
    # Small samples: 5-fold cross-validation
    # ======================================================================
    if use_cv:
        print(f'n = {n_total} < {SMALL_SAMPLE_THRESHOLD}: 5-fold cross-validation.')
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # SMOTE inside each fold, so synthetic minority points are never derived
        # from records that end up in that fold's validation set.
        cv_pipelines = {
            'Logistic Regression': ImbPipeline([
                ('smote', SMOTE(random_state=42)),
                ('clf', LogisticRegression(max_iter=1000, random_state=42,
                                           class_weight='balanced'))]),
            'Random Forest': ImbPipeline([
                ('smote', SMOTE(random_state=42)),
                ('clf', RandomForestClassifier(n_estimators=100, max_depth=10,
                                               random_state=42,
                                               class_weight='balanced'))]),
            'LightGBM': ImbPipeline([
                ('smote', SMOTE(random_state=42)),
                ('clf', LGBMClassifier(random_state=42, verbose=-1,
                                       class_weight='balanced'))]),
        }
        scoring = {'accuracy': 'accuracy', 'precision': 'precision',
                   'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}

        rows = []
        for name, pipe in cv_pipelines.items():
            res = cross_validate(pipe, X_encoded, y, cv=cv,
                                 scoring=scoring, n_jobs=-1)
            rows.append({
                'Model': name,
                'Accuracy':  f"{res['test_accuracy'].mean():.3f} ±{res['test_accuracy'].std():.3f}",
                'Precision': f"{res['test_precision'].mean():.3f} ±{res['test_precision'].std():.3f}",
                'Recall':    f"{res['test_recall'].mean():.3f} ±{res['test_recall'].std():.3f}",
                'F1':        f"{res['test_f1'].mean():.3f} ±{res['test_f1'].std():.3f}",
                'AUC':       f"{res['test_roc_auc'].mean():.3f} ±{res['test_roc_auc'].std():.3f}",
                'Eval': '5-fold CV',
            })
            print(f"  {name}: AUC={res['test_roc_auc'].mean():.3f}"
                  f"±{res['test_roc_auc'].std():.3f}")
        all_region_metrics[region] = pd.DataFrame(rows)

        # -- mean ROC across folds; same cv object, so folds match the table ---
        mean_fpr = np.linspace(0, 1, 200)
        model_specs = {
            'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42,
                                                      class_weight='balanced'),
            'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10,
                                                    random_state=42,
                                                    class_weight='balanced'),
            'LightGBM': LGBMClassifier(random_state=42, verbose=-1,
                                       class_weight='balanced'),
        }
        colors = {'Logistic Regression': '#1f77b4',
                  'Random Forest': '#ff7f0e',
                  'LightGBM': '#2ca02c'}
        splits = list(cv.split(X_encoded, y))

        plt.figure(figsize=(9, 6))
        for name, base_model in model_specs.items():
            tprs, fold_aucs = [], []
            for tr, te in splits:
                X_tr, y_tr = X_encoded.iloc[tr], y.iloc[tr]
                X_te, y_te = X_encoded.iloc[te], y.iloc[te]
                X_res, y_res = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
                m = copy.deepcopy(base_model).fit(X_res, y_res)
                prob = m.predict_proba(X_te)[:, 1]
                f, t, _ = roc_curve(y_te, prob)
                fold_aucs.append(auc(f, t))
                tprs.append(np.interp(mean_fpr, f, t))
            plt.plot(mean_fpr, np.mean(tprs, axis=0), color=colors[name], linewidth=2,
                     label=f'{name} (AUC={np.mean(fold_aucs):.3f} ±{np.std(fold_aucs):.3f})')
        plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve — {region}\n(5-fold mean, SMOTE inside each fold)')
        plt.legend(loc='lower right'); plt.grid(alpha=0.3); plt.tight_layout()
        plt.savefig(f'{slug}_roc.png', dpi=300); plt.show()

        print(f'{region}: skipping SHAP (fold-level model instability makes a '
              f'feature ranking non-reproducible at this sample size).')

    # ======================================================================
    # Large sample: single 80/20 split, plus SHAP
    # ======================================================================
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X_encoded, y, test_size=0.2, random_state=42, stratify=y)
        print(f'80/20 split: {len(X_train)} train, {len(X_test)} test')

        # SMOTE fitted on the training partition only; the test set keeps its
        # natural class distribution.
        X_train_res, y_train_res = SMOTE(random_state=42).fit_resample(X_train, y_train)

        lr_model = LogisticRegression(max_iter=1000, random_state=42,
                                      class_weight='balanced').fit(X_train_res, y_train_res)
        lgb_model = LGBMClassifier(random_state=42, verbose=-1,
                                   class_weight='balanced').fit(X_train_res, y_train_res)

        # Random Forest tuning: SMOTE lives inside the pipeline, so it is applied
        # separately within each search fold and never leaks synthetic points from
        # a validation fold into its own training data. Fit on the RAW training
        # partition, not the pre-oversampled one.
        cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        rf_pipeline = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', RandomForestClassifier(random_state=42, class_weight='balanced')),
        ])
        grid = GridSearchCV(
            rf_pipeline,
            {'clf__n_estimators': [100, 200], 'clf__max_depth': [10, 20, None]},
            cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_rf = grid.best_estimator_          # a fitted ImbPipeline
        print(f'Random Forest best parameters: {grid.best_params_}')
        print(f'Random Forest best CV AUC: {grid.best_score_:.3f}')

        final_models = {'Logistic Regression': lr_model,
                        'Random Forest': best_rf,
                        'LightGBM': lgb_model}

        rows = []
        plt.figure(figsize=(9, 6))
        for name, model in final_models.items():
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)[:, 1]
            f, t, _ = roc_curve(y_test, y_prob)
            roc_auc = auc(f, t)
            rows.append({
                'Model': name,
                'Accuracy':  round(accuracy_score(y_test, y_pred), 3),
                'Precision': round(precision_score(y_test, y_pred, zero_division=0), 3),
                'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 3),
                'F1':        round(f1_score(y_test, y_pred, zero_division=0), 3),
                'AUC':       round(roc_auc, 3),
                'Eval': '80/20 split',
            })
            plt.plot(f, t, label=f'{name} (AUC={roc_auc:.3f})', linewidth=2)
        all_region_metrics[region] = pd.DataFrame(rows)

        plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve — {region}\n'
                  f'(leakage-free features, SMOTE on training set only)')
        plt.legend(loc='lower right'); plt.grid(alpha=0.3); plt.tight_layout()
        plt.savefig(f'{slug}_roc.png', dpi=300); plt.show()

        # ---- SHAP (Greater London only) --------------------------------------
        # Computed on a random subsample of up to 500 held-out test records to
        # keep TreeExplainer runtime manageable. The seed is fixed and the draw
        # ignores the outcome variable, so it is unbiased with respect to severity.
        X_shap = X_test.sample(min(500, len(X_test)), random_state=42)
        print(f'SHAP computed on {len(X_shap)} of {len(X_test)} test records')

        shap_models = {
            'RandomForest': best_rf.named_steps['clf'],   # unwrap the pipeline
            'LightGBM':     lgb_model,
        }
        for model_name, estimator in shap_models.items():
            try:
                explainer = shap.TreeExplainer(estimator)
                sv = explainer.shap_values(X_shap)
                # normalise the return shape across SHAP versions / model types
                if isinstance(sv, list):
                    sv = sv[1]
                elif isinstance(sv, np.ndarray) and sv.ndim == 3:
                    sv = sv[:, :, 1]

                plt.figure(figsize=(10, 6))
                shap.summary_plot(sv, X_shap, plot_type='bar', show=False, max_display=10)
                plt.title(f'SHAP Feature Importance — {region} ({model_name})')
                plt.tight_layout()
                plt.savefig(f'{slug}_shap_bar_{model_name.lower()}.png',
                            dpi=300, bbox_inches='tight')
                plt.show()

                plt.figure(figsize=(10, 6))
                shap.summary_plot(sv, X_shap, show=False, max_display=10)
                plt.title(f'SHAP Beeswarm — {region} ({model_name})')
                plt.tight_layout()
                plt.savefig(f'{slug}_shap_beeswarm_{model_name.lower()}.png',
                            dpi=300, bbox_inches='tight')
                plt.show()

                # Dependence plot for sex_of_casualty: fixes the direction of the
                # sex effect, which the beeswarm shows only as a colour gradient.
                sex_col = [c for c in X_shap.columns if 'sex_of_casualty' in c]
                if sex_col:
                    plt.figure(figsize=(8, 5))
                    shap.dependence_plot(sex_col[0], sv, X_shap,
                                         interaction_index=None, show=False)
                    plt.title(f'SHAP Dependence - {sex_col[0]}\n{region} ({model_name})')
                    plt.tight_layout()
                    plt.savefig(f'{slug}_shap_dependence_sex_{model_name.lower()}.png',
                                dpi=300, bbox_inches='tight')
                    plt.show()

            except Exception as e:
                print(f'WARNING: SHAP failed ({region}/{model_name}): {e}')

    print(f'\n{region} model performance:')
    print(all_region_metrics[region].to_string(index=False))

# ==========================================================================
# Summary across the three study regions
# ==========================================================================
print('\n\n========== Model performance summary, three study regions ==========')
for region, metrics in all_region_metrics.items():
    print(f'\n--- {region} ---')
    print(metrics.to_string(index=False))

pd.concat([m.assign(Region=r) for r, m in all_region_metrics.items()]
          ).to_csv('tables_5_to_7_model_performance.csv', index=False)
print('\nSaved: tables_5_to_7_model_performance.csv')


In [ ]:
# Cell D is not needed to reproduce Chapter 5. It refits the same three
# classifiers with every STATS19 code field one-hot encoded and -1 kept as an
# explicit level, and prints the comparison reported in Appendix D of the report.
#
# Its figures and table are written to separate filenames (*_categorical.png and
# appendix_d_categorical_encoding.csv), so running it leaves the Chapter 5
# outputs from Cell C untouched.

"""
Cell D — OPTIONAL: categorical-encoding sensitivity check (Appendix D)
==============================================
Produces Tables 5-7, Figures 7-9 (ROC) and Figures 10-12 (SHAP).

Encoding note (this version):
    STATS19 code fields are unordered classifications, not quantities. Every one
    of the sixteen candidate features arrives from the CSV as int64, so the
    previous `get_dummies(X, drop_first=True)` call found no object columns and
    did nothing: each code entered the model as a raw number, and the -1 code
    ("missing / unknown / not applicable") sat one step below the lowest valid
    value. For casualty_imd_decile that placed 1,113 casualties with no
    residential LSOA below Decile 1, i.e. below the most deprived category.
    This version one-hot encodes the code fields so that -1 becomes an explicit
    level, keeps speed_limit and the counts numeric, and handles the 371
    unknown ages with a missingness flag plus median imputation.

Other properties, unchanged from the previous version:
1. All three regions use the Combined Authority extracts, so the modelling and
   the exposure analysis in Cell A cover the same geography.
2. The Random Forest grid search wraps SMOTE and the classifier in an
   imbalanced-learn Pipeline, so oversampling happens inside each search fold.
3. sex_of_casualty == -1 is filtered out before modelling.
4. SHAP is run for Greater London only, for estimate stability rather than
   headline AUC.
"""

import copy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, GridSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve, auc)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------- configuration
# Reuses the REGION_FILES resolved in Cell A if that cell has already run;
# otherwise resolves them the same way.
try:
    REGION_FILES
except NameError:
    import glob, os
    def _first(*pats):
        roots = ['.', os.path.expanduser('~/Downloads'), os.path.expanduser('~/Desktop')]
        for root in roots:
            for p in pats:
                hits = sorted(glob.glob(os.path.join(root, '**', p), recursive=True))
                if hits:
                    return hits[0]
        raise FileNotFoundError(f'none of {pats} found under {roots}')
    REGION_FILES = {
        'Greater London':     _first('london_point*.csv', '*london*point*.csv'),
        'Greater Manchester': _first('manchester_point*.csv', '*manchester*point*.csv'),
        'West Midlands':      _first('birmingham_point*.csv', '*birmingham*point*.csv'),
    }

SMALL_SAMPLE_THRESHOLD = 1000    # below this n, use 5-fold cross-validation

# Feature selection is a positive whitelist rather than an exclusion list: only
# the variables named here can enter the model, so every post-crash STATS19
# descriptor (pedestrian_movement, vehicle_leaving_carriageway,
# hit_object_in_carriageway, ...) is excluded by construction.
CANDIDATE_FEATURES = [
    'casualty_imd_decile',
    'speed_limit',
    'road_type',
    'junction_detail',
    'light_conditions',
    'weather_conditions',
    'road_surface_conditions',
    'day_of_week',
    'first_road_class',
    'age_of_casualty',
    'sex_of_casualty',
    'casualty_type',          # single-valued in a cyclist-only sample
    'pedestrian_location',    # single-valued in a cyclist-only sample
    'urban_or_rural_area',
    'number_of_vehicles',
    'number_of_casualties',
]

# Unordered STATS19 classifications. One-hot encoded, so -1 becomes its own
# level instead of a number below the lowest valid code. casualty_type and
# pedestrian_location take one value each here, so drop_first removes them.
CATEGORICAL = ['casualty_imd_decile', 'road_type', 'junction_detail',
               'light_conditions', 'weather_conditions', 'road_surface_conditions',
               'day_of_week', 'first_road_class', 'sex_of_casualty',
               'casualty_type', 'pedestrian_location', 'urban_or_rural_area']

# speed_limit is an ordered scale in mph (20-70) with no missing values;
# the remaining three are genuine counts or ages.
NUMERIC = ['speed_limit', 'age_of_casualty', 'number_of_vehicles', 'number_of_casualties']

all_region_metrics = {}
shap_store = {}          # keeps the SHAP output for the decomposition at the end

for region, path in REGION_FILES.items():
    # Cell D writes to *_categorical.png so it cannot overwrite the figures
    # Cell C produced, which are the ones reported in Chapter 5.
    slug = region.lower().replace(' ', '_') + '_categorical'
    print(f"\n{'='*60}\n{region}\n{'='*60}")

    df = pd.read_csv(path, low_memory=False)

    # ---------- target: STATS19 1 = Fatal, 2 = Serious, 3 = Slight ------------
    df['Y'] = np.where(df['collision_severity'] <= 2, 1, 0)

    # ---------- drop sex_of_casualty == -1 only -------------------------------
    # The STATS19 "unknown" sex code is removed rather than kept as a level,
    # because it produced attributions unlike either male or female. The -1 code
    # is retained as an explicit level for every other feature.
    if 'sex_of_casualty' in df.columns:
        before = len(df)
        df = df[df['sex_of_casualty'] != -1].reset_index(drop=True)
        print(f'Removed sex_of_casualty == -1: {before} -> {len(df)} '
              f'({before - len(df)} rows)')

    n_total = len(df)
    print(f'Sample size: {n_total};  serious/fatal share: {df["Y"].mean():.1%}')

    # ---------- leakage guard -------------------------------------------------
    feature_cols = [c for c in CANDIDATE_FEATURES if c in df.columns]
    leak = [c for c in feature_cols if 'severity' in c.lower()]
    if leak:
        raise ValueError(f'Leakage guard tripped: {leak}')

    # ---------- encoding ------------------------------------------------------
    X = df[[c for c in CATEGORICAL + NUMERIC if c in df.columns]].copy()

    # age_of_casualty == -1 means unknown, not an age. Flag it, then impute the
    # median, so the flag carries the missingness and the age column carries
    # only real ages.
    if 'age_of_casualty' in X.columns:
        miss = X['age_of_casualty'] == -1
        X['age_missing'] = miss.astype(int)
        X.loc[miss, 'age_of_casualty'] = np.nan
        X['age_of_casualty'] = X['age_of_casualty'].fillna(X['age_of_casualty'].median())
        print(f'age_of_casualty == -1: {int(miss.sum())} flagged and median-imputed')

    cat_present = [c for c in CATEGORICAL if c in X.columns]
    for c in cat_present:
        print(f'  {c:<26} levels: {sorted(X[c].unique().tolist())}')
    X[cat_present] = X[cat_present].astype(int).astype(str)
    X_encoded = pd.get_dummies(X, columns=cat_present, drop_first=True)
    y = df['Y']
    print(f'Encoded feature matrix: {X_encoded.shape[1]} columns '
          f'from {len(feature_cols)} raw features')

    use_cv = n_total < SMALL_SAMPLE_THRESHOLD

    # ======================================================================
    # Small samples: 5-fold cross-validation
    # ======================================================================
    if use_cv:
        print(f'n = {n_total} < {SMALL_SAMPLE_THRESHOLD}: 5-fold cross-validation.')
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # SMOTE inside each fold, so synthetic minority points are never derived
        # from records that end up in that fold's validation set.
        cv_pipelines = {
            'Logistic Regression': ImbPipeline([
                ('smote', SMOTE(random_state=42)),
                ('clf', LogisticRegression(max_iter=1000, random_state=42,
                                           class_weight='balanced'))]),
            'Random Forest': ImbPipeline([
                ('smote', SMOTE(random_state=42)),
                ('clf', RandomForestClassifier(n_estimators=100, max_depth=10,
                                               random_state=42,
                                               class_weight='balanced'))]),
            'LightGBM': ImbPipeline([
                ('smote', SMOTE(random_state=42)),
                ('clf', LGBMClassifier(random_state=42, verbose=-1,
                                       class_weight='balanced'))]),
        }
        scoring = {'accuracy': 'accuracy', 'precision': 'precision',
                   'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}

        rows = []
        for name, pipe in cv_pipelines.items():
            res = cross_validate(pipe, X_encoded, y, cv=cv,
                                 scoring=scoring, n_jobs=-1)
            rows.append({
                'Model': name,
                'Accuracy':  f"{res['test_accuracy'].mean():.3f} ±{res['test_accuracy'].std():.3f}",
                'Precision': f"{res['test_precision'].mean():.3f} ±{res['test_precision'].std():.3f}",
                'Recall':    f"{res['test_recall'].mean():.3f} ±{res['test_recall'].std():.3f}",
                'F1':        f"{res['test_f1'].mean():.3f} ±{res['test_f1'].std():.3f}",
                'AUC':       f"{res['test_roc_auc'].mean():.3f} ±{res['test_roc_auc'].std():.3f}",
                'Eval': '5-fold CV',
            })
            print(f"  {name}: AUC={res['test_roc_auc'].mean():.3f}"
                  f"±{res['test_roc_auc'].std():.3f}")
        all_region_metrics[region] = pd.DataFrame(rows)

        # -- mean ROC across folds; same cv object, so folds match the table ---
        mean_fpr = np.linspace(0, 1, 200)
        model_specs = {
            'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42,
                                                      class_weight='balanced'),
            'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10,
                                                    random_state=42,
                                                    class_weight='balanced'),
            'LightGBM': LGBMClassifier(random_state=42, verbose=-1,
                                       class_weight='balanced'),
        }
        colors = {'Logistic Regression': '#1f77b4',
                  'Random Forest': '#ff7f0e',
                  'LightGBM': '#2ca02c'}
        splits = list(cv.split(X_encoded, y))

        plt.figure(figsize=(9, 6))
        for name, base_model in model_specs.items():
            tprs, fold_aucs = [], []
            for tr, te in splits:
                X_tr, y_tr = X_encoded.iloc[tr], y.iloc[tr]
                X_te, y_te = X_encoded.iloc[te], y.iloc[te]
                X_res, y_res = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
                m = copy.deepcopy(base_model).fit(X_res, y_res)
                prob = m.predict_proba(X_te)[:, 1]
                f, t, _ = roc_curve(y_te, prob)
                fold_aucs.append(auc(f, t))
                tprs.append(np.interp(mean_fpr, f, t))
            plt.plot(mean_fpr, np.mean(tprs, axis=0), color=colors[name], linewidth=2,
                     label=f'{name} (AUC={np.mean(fold_aucs):.3f} ±{np.std(fold_aucs):.3f})')
        plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve — {region}\n(5-fold mean, SMOTE inside each fold)')
        plt.legend(loc='lower right'); plt.grid(alpha=0.3); plt.tight_layout()
        plt.savefig(f'{slug}_roc.png', dpi=300); plt.show()

        print(f'{region}: skipping SHAP (fold-level model instability makes a '
              f'feature ranking non-reproducible at this sample size).')

    # ======================================================================
    # Large sample: single 80/20 split, plus SHAP
    # ======================================================================
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X_encoded, y, test_size=0.2, random_state=42, stratify=y)
        print(f'80/20 split: {len(X_train)} train, {len(X_test)} test')

        # SMOTE fitted on the training partition only; the test set keeps its
        # natural class distribution.
        X_train_res, y_train_res = SMOTE(random_state=42).fit_resample(X_train, y_train)

        lr_model = LogisticRegression(max_iter=1000, random_state=42,
                                      class_weight='balanced').fit(X_train_res, y_train_res)
        lgb_model = LGBMClassifier(random_state=42, verbose=-1,
                                   class_weight='balanced').fit(X_train_res, y_train_res)

        # Random Forest tuning: SMOTE lives inside the pipeline, so it is applied
        # separately within each search fold and never leaks synthetic points from
        # a validation fold into its own training data. Fit on the RAW training
        # partition, not the pre-oversampled one.
        cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        rf_pipeline = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', RandomForestClassifier(random_state=42, class_weight='balanced')),
        ])
        grid = GridSearchCV(
            rf_pipeline,
            {'clf__n_estimators': [100, 200], 'clf__max_depth': [10, 20, None]},
            cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
        grid.fit(X_train, y_train)
        best_rf = grid.best_estimator_          # a fitted ImbPipeline
        print(f'Random Forest best parameters: {grid.best_params_}')
        print(f'Random Forest best CV AUC: {grid.best_score_:.3f}')

        final_models = {'Logistic Regression': lr_model,
                        'Random Forest': best_rf,
                        'LightGBM': lgb_model}

        rows = []
        plt.figure(figsize=(9, 6))
        for name, model in final_models.items():
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)[:, 1]
            f, t, _ = roc_curve(y_test, y_prob)
            roc_auc = auc(f, t)
            rows.append({
                'Model': name,
                'Accuracy':  round(accuracy_score(y_test, y_pred), 3),
                'Precision': round(precision_score(y_test, y_pred, zero_division=0), 3),
                'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 3),
                'F1':        round(f1_score(y_test, y_pred, zero_division=0), 3),
                'AUC':       round(roc_auc, 3),
                'Eval': '80/20 split',
            })
            plt.plot(f, t, label=f'{name} (AUC={roc_auc:.3f})', linewidth=2)
        all_region_metrics[region] = pd.DataFrame(rows)

        plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve — {region}\n'
                  f'(leakage-free features, SMOTE on training set only)')
        plt.legend(loc='lower right'); plt.grid(alpha=0.3); plt.tight_layout()
        plt.savefig(f'{slug}_roc.png', dpi=300); plt.show()

        # ---- SHAP (Greater London only) --------------------------------------
        # Computed on a random subsample of up to 500 held-out test records to
        # keep TreeExplainer runtime manageable. The seed is fixed and the draw
        # ignores the outcome variable, so it is unbiased with respect to severity.
        X_shap = X_test.sample(min(500, len(X_test)), random_state=42)
        print(f'SHAP computed on {len(X_shap)} of {len(X_test)} test records')

        shap_models = {
            'RandomForest': best_rf.named_steps['clf'],   # unwrap the pipeline
            'LightGBM':     lgb_model,
        }
        for model_name, estimator in shap_models.items():
            try:
                explainer = shap.TreeExplainer(estimator)
                sv = explainer.shap_values(X_shap)
                # normalise the return shape across SHAP versions / model types
                if isinstance(sv, list):
                    sv = sv[1]
                elif isinstance(sv, np.ndarray) and sv.ndim == 3:
                    sv = sv[:, :, 1]

                if model_name == 'LightGBM':
                    shap_store[region] = (sv, X_shap, df)   # kept for the decomposition

                plt.figure(figsize=(10, 6))
                shap.summary_plot(sv, X_shap, plot_type='bar', show=False, max_display=10)
                plt.title(f'SHAP Feature Importance — {region} ({model_name})')
                plt.tight_layout()
                plt.savefig(f'{slug}_shap_bar_{model_name.lower()}.png',
                            dpi=300, bbox_inches='tight')
                plt.show()

                plt.figure(figsize=(10, 6))
                shap.summary_plot(sv, X_shap, show=False, max_display=10)
                plt.title(f'SHAP Beeswarm — {region} ({model_name})')
                plt.tight_layout()
                plt.savefig(f'{slug}_shap_beeswarm_{model_name.lower()}.png',
                            dpi=300, bbox_inches='tight')
                plt.show()

                # Dependence plot for sex_of_casualty. After one-hot encoding the
                # column is named sex_of_casualty_2 (male, code 1, is the dropped
                # reference level), so a value of 1 on this axis means female.
                sex_col = [c for c in X_shap.columns if c.startswith('sex_of_casualty')]
                if sex_col:
                    plt.figure(figsize=(8, 5))
                    shap.dependence_plot(sex_col[0], sv, X_shap,
                                         interaction_index=None, show=False)
                    plt.title(f'SHAP Dependence — {sex_col[0]}\n{region} ({model_name})')
                    plt.tight_layout()
                    plt.savefig(f'{slug}_shap_dependence_sex_{model_name.lower()}.png',
                                dpi=300, bbox_inches='tight')
                    plt.show()

            except Exception as e:
                print(f'WARNING: SHAP failed ({region}/{model_name}): {e}')

    print(f'\n{region} model performance:')
    print(all_region_metrics[region].to_string(index=False))

# ==========================================================================
# Summary across the three study regions
# ==========================================================================
print('\n\n========== Model performance summary, three study regions ==========')
for region, metrics in all_region_metrics.items():
    print(f'\n--- {region} ---')
    print(metrics.to_string(index=False))

pd.concat([m.assign(Region=r) for r, m in all_region_metrics.items()]
          ).to_csv('appendix_d_categorical_encoding.csv', index=False)
print('\nSaved: appendix_d_categorical_encoding.csv')

# ==========================================================================
# SHAP decomposition by raw STATS19 code  (Appendix C and Section 5.4)
# ==========================================================================
# One-hot encoding splits a code field across several dummy columns, and
# drop_first removes the reference level entirely. To recover what the model
# attributes to the field as a whole, the SHAP values of all of that field's
# dummy columns are summed for each record, then grouped by the record's
# original raw code. Records in the dropped reference level correctly receive a
# total of zero from the dummies, so their contribution is read relative to it.

def decompose_by_code(field, sv, X_shap, source_df):
    cols = [i for i, c in enumerate(X_shap.columns) if c.startswith(field + '_')]
    if not cols:
        print(f'{field}: no dummy columns found')
        return None
    total = sv[:, cols].sum(axis=1)
    out = pd.DataFrame({
        'code': source_df.loc[X_shap.index, field].values,
        'shap': total,
    })
    table = (out.groupby('code')['shap']
             .agg(n='count', mean='mean', sd='std')
             .round(3)
             .reset_index())
    table['share'] = (table['n'] / table['n'].sum() * 100).round(1)
    return table[['code', 'n', 'share', 'mean', 'sd']]

if 'Greater London' in shap_store:
    sv_gl, X_shap_gl, df_gl = shap_store['Greater London']
    print('\n\n========== SHAP by raw code — Greater London, LightGBM ==========')
    for field in ['road_surface_conditions', 'casualty_imd_decile',
                  'junction_detail', 'first_road_class']:
        t = decompose_by_code(field, sv_gl, X_shap_gl, df_gl)
        if t is not None:
            print(f'\n--- {field} ---')
            print(t.to_string(index=False))

    # mean |SHAP| per original feature, so the ranking can be reported at the
    # level of the variable rather than of individual dummy columns
    print('\n--- mean |SHAP| aggregated to the original feature ---')
    agg = {}
    for feat in CANDIDATE_FEATURES + ['age_missing']:
        cols = [i for i, c in enumerate(X_shap_gl.columns)
                if c == feat or c.startswith(feat + '_')]
        if cols:
            agg[feat] = np.abs(sv_gl[:, cols].sum(axis=1)).mean()
    rank = pd.Series(agg).sort_values(ascending=False).round(3)
    print(rank.to_string())
